# Trabalho IA - Dados Atividades Agrícolas

**Nome do Arquivo:** Dados_agricultura_AtividadeIA.ipynb  
**Autores:**
- DANIELA MARCHETTI CAMPOS RODRIGUES - 2024015808
- ERICK LUZ AQUINO - 2021017219
- HANNAH CLARA SILVA OLIVEIRA - 2023009163
- IGOR DINIZ SILVA - 2024016028
- MARCOS PAULO CYRILLO DA SILVA - 2024015871

**Descrição:** Este trabalho utiliza o conjunto de dados Crop Recommendation para analisar características do solo e do clima, com o objetivo de recomendar o tipo de cultivo mais adequado. Foram realizadas as etapas de tratamento dos dados, agrupamento utilizando o modelo K-Means e classificação por meio do modelo MLP. Ao final, são apresentados e discutidos os resultados obtidos.

# 1. Obtenção e tratamento dos dados

---



---



## 1.1 Obtenção dos dados
**Objetivo:**
O conjunto de dados visa recomendar o tipo de cultura (plantação) mais apropriado a partir de características ambientais e de solo. Ele é amplamente usado em tarefas de classificação supervisionada em aprendizado de máquina.

**Composição dos Dados:**
- Número total de instâncias: 2.200 amostras
- Número de atributos (features): 7 variáveis independentes + 1 variável alvo

**Atributos:**
| Atributo | Descrição |
|---------|-----------|
| **N** | Concentração de Nitrogênio no solo (ppm) |
| **P** | Concentração de Fósforo no solo (ppm) |
| **K** | Concentração de Potássio no solo (ppm) |
| **temperature** | Temperatura média do ambiente (°C) |
| **humidity** | Umidade relativa do ar (%) |
| **ph** | Índice de acidez do solo |
| **rainfall** | Precipitação média anual (mm) |
| **label** | Cultura recomendada (variável alvo, categórica) |


**Número de classes (culturas):**
22 tipos diferentes de culturas (ex.: arroz, milho, algodão, café, etc.)

**Distribuição de amostras por classe:**
Cada cultura possui aproximadamente 100 amostras, o que torna o conjunto de dados equilibrado — ideal para tarefas de classificação supervisionada com algoritmos como o MLPClassifier.

O link para consulta dos datasets é apresentado a seguir.

**Link:** https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset

## 1.2 Tratamento dos dados
O tratamento dos dados seguiu as etapas de visualização inicial, para compreender como eles estavam organizados. Em seguida, realizou-se a etapa de tratamento e limpeza, com a finalidade de filtrar apenas as informações relevantes para o presente projeto. Por fim, foram realizadas as etapas de transformação dos dados para seu emprego no modelo KNN.

### 1.2.1 Importando as bibliotecas, pacotes e realizando o carregamento do dataframe

In [50]:
#Caso esteja rodando localmente e não possuir as seguintes bibliotecas instaladas: pip install pandas numpy scikit-learn matplotlib
#Aqui importei a biblioteca pandas e data= foi utlizado os dados ja adicionados do csv.
#Em seguida os dados de cabeçalho, informação e descrição foram impressos em tela conforme abaixo.


import pandas as pd
import numpy as np

"""
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/Trabalho IA"
"""
from google.colab import files
files.upload()

data = pd.read_csv("Crop_recommendation.csv")
data.head()
data.info()
data.describe()

Saving Crop_recommendation.csv to Crop_recommendation (7).csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   N            2200 non-null   int64 
 1   P            2200 non-null   int64 
 2   K            2200 non-null   int64 
 3   temperature  2200 non-null   object
 4   humidity     2200 non-null   object
 5   ph           2200 non-null   object
 6   rainfall     2200 non-null   object
 7   label        2200 non-null   object
dtypes: int64(3), object(5)
memory usage: 137.6+ KB


,N,P,K
count,2200.000000,2200.000000,2200.000000
mean,50.551818,53.362727,48.149091
std,36.917334,32.985883,50.647931
min,0.000000,5.000000,5.000000
25%,21.000000,28.000000,20.000000
50%,37.000000,51.000000,32.000000
75%,84.250000,68.000000,49.000000
max,140.000000,145.000000,205.000000


Sobre as bibliotecas e pacotes utilizados:
* **pandas:** utilizado para a manipulação e análise dos dados dos datasets.
* **pathlib:** utilizado para o gerenciamento dos caminhos dos arquivos dos datasets e dos arquivos de saída.
* **csr_matrix:** utilizado para a criação de uma matriz esparsa após o tratamento dos dados.
* **NearestNeighbors:** utilizado para o treinamento do modelo KNN.
* **joblib:** utilizado para o salvamento do modelo treinado e da matriz pivot tratada, possibilitando o uso dessas informações em outros ambientes de desenvolvimento.

In [52]:
data.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,2.087.974.371,8.200.274.423,6.502.985.292.000.000,2.029.355.362,rice
1,85,58,41,2.177.046.169,8.031.964.408,7.038.096.361,2.266.555.374,rice
2,60,55,44,2.300.445.915,823.207.629,7.840.207.144,2.639.642.476,rice
3,74,35,40,2.649.109.635,8.015.836.264,6.980.400.905,2.428.640.342,rice
4,78,42,42,2.013.017.482,8.160.487.287,7.628.472.891,2.627.173.405,rice


### 1.2.2 Verificação de dados inconsistentes

In [53]:
# Esse método foi usado para verificar dados inconsistentes;
data.isnull().sum()


,0
N,0
P,0
K,0
temperature,0
humidity,0
ph,0
rainfall,0
label,0


Logo, não possuimos nenhum valor nulo em nosso dataset

### 1.2.3 Codificação de Variáveis Categóricas

Como a variável alvo (label) contém nomes de culturas (strings), é necessário convertê-la para valores numéricos antes de usar modelos do scikit-learn:

In [54]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
data['label_encoded'] = encoder.fit_transform(data['label'])

### 1.2.4 Padronização (Escalonamento)

Os atributos numéricos variam em escalas diferentes (por exemplo, pH de 0–14 e temperatura em °C). Isso pode prejudicar algoritmos sensíveis à escala, como K-Means e MLP.
Portanto, aplicamos padronização:

In [55]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = data.drop(['label', 'label_encoded'], axis=1)
X_scaled = scaler.fit_transform(X)

ValueError: could not convert string to float: '2.087.974.371'

Aqui nessa parte foi realizada a separação para teste (no caso de 20%);

In [ ]:
from sklearn.model_selection import train_test_split

y = data['label_encoded']
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

Basicamente utilizamos a padronização Z-Score, que tranforma todos os atributos (numéricos) para média 0  e desvio padrão 1

In [ ]:
data.head()

# 2. Modelo K-Médias

##2.1 Características do K-Médias


1.   Modelo de aprendizado não supervisionado
2.   Número de clusters definido previamente
3.   Busca pela menor inércia (distância de um ponto até o cluster mais próximo)



##2.2 Como funciona o agrupamento usando o modelo K-Médias?

A partir de um grupo de dados X = {a, b, c, ..., n}, o algoritmo (ou o usuário) escolhe centros, pontos que servem de referência para o cálculo das distâncias e aos quais serão associados os pontos do conjunto X mais próximos. Depois de serem designados a um grupo, novos centros são escolhidos e o algoritmo se repete até uma parada pré-estabelecida ou até que não haja mais mudança nos centros.

In [ ]:
#BIBLIOTECAS INDISPENSÁVEIS
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

##2.3 Analisando a função KMeans do sklearn.cluster

KMeans(n_clusters, init, n_init, max_iter, tol, precompute_distances, verbose, random_state, copy_x, n_jobs, algorithm)

Os campos mais importantes nesse instante são:


*   n_clusters (define quantos grupos e centróides serão formados)
*   n_init (quantas vezes o KMeans será inicializado, com diferentes centróides, para entregar a melhor execução inicial possível)

Além desses, outro parâmetro que será explorado:

*   max_iter (número máximo de iterações que o algoritmo realiza por execução)

##2.4 Definindo a quantidade ideal de clusters

A fim de descobrir qual o melhor número de clusters que se pode definir para a função KMeans da biblioteca sklearn.cluster, podemos aplicar o método elbow.

In [ ]:
#Função para encontrar o melhor número de clusters
def kmeansElbow(data, max):
  qtdClusters = []
  inercias = []

  for k in range(1, max):
    kmeans = KMeans(n_clusters = k, random_state=42)
    kmeans.fit(data)

    qtdClusters.append(k)
    inercias.append(kmeans.inertia_)

  fig = plt.subplots(figsize=(10, 5))
  plt.plot(qtdClusters, inercias)
  plt.xlabel('Qtd de clusters')
  plt.ylabel('Inércia')
  plt.grid(True)
  plt.show()

In [ ]:
kmeansElbow(X_scaled, 21)

Observando o gráfico, podemos implicar que 11 seria o número ideal que buscamos porque não há evolução significativa na inércia além dessa quantidade de grupos.

##2.5 Treino e teste do modelo com 11 clusters





In [ ]:
kmeans = KMeans(n_clusters=11, random_state=42)
kmeans.fit(X_train, y_train)

In [ ]:
#Criando dataframes para ser possível inserir uma coluna indicando o cluster de cada dado
trainKM = pd.DataFrame(X_train, columns=X.columns)
testKM = pd.DataFrame(X_test, columns=X.columns)

trainClusters = kmeans.predict(X_train)
testClusters = kmeans.predict(X_test)


trainKM['CLUSTER'] = trainClusters + 1
testKM['CLUSTER'] = testClusters + 1


In [ ]:
#Visualizando como fica a nova coluna
print(f'Porção treino:\n---\n{trainKM['CLUSTER'].head()}\n')
print(f'Porção teste:\n----\n{testKM['CLUSTER'].head()}')

## 2.6 Visualização dos resultados no teste

###Medidas de desempenho

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score, v_measure_score

ari = adjusted_rand_score(y_test, testClusters)
nmi = normalized_mutual_info_score(y_test, testClusters)
homogeneity = homogeneity_score(y_test, testClusters)
completeness = completeness_score(y_test, testClusters)
v_measure = v_measure_score(y_test, testClusters)

print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi:.4f}")
print(f"Homogeneidade: {homogeneity:.4f}")
print(f"Completude {completeness:.4f}")
print(f"V-measure: {v_measure:.4f}")
print(f"-------------------------------------")

####**ARI**
Medida que varia de -1 a 1 usada para encontrar o grau de semelhança entre pares de pontos, dada por:

$$ ARI = \frac{RI - E}{1 - E}$$

Sendo:
* RI: Rand Index
* E: Valor esperado para Rand Index

O valor 0.45 (45%) nos indica que a partição não foi ideal, mas chegou próxima de uma clusterização mediana.
____
####**NMI**
Medida que varia de 0 a 1, avaliando o grau de semelhança geral entre clusters

O valor 0.75 (75%) indica que grande parte das culturas semelhantes entraram nos mesmos grupos.
____
####Homogeneidade
Medida que varia de 0 a 1, revelando o quão semelhantes são os pontos dentro um mesmo grupo.

O valor 0.65 (65%) sugere uma semelhança acima da média entre os pontos dos clusters, ainda que não ideal.
____
####**Completude**
Medida que varia de 0 a 1, avaliando se os membros similares estão em um mesmo grupo.

O valor 0.88 (88%) indica que grande parte dos pontos semelhantes entraram nos grupos de seus "parceiros".
____
####**V-measure**
Medida que varia de 0 a 1 e combina completude e homogeneidade.

O valor 0.75 sugere que o algoritmo encontrou uma boa estrutura entre os componentes dos grupos e seus similares.

### Distribuição dos resultados

In [ ]:
pcaTest11 = testKM

#pegando as colunas numéricas de testKM
colNumTest = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X_testNum = pcaTest11[colNumTest]

#pré-processamento dos dados
pcaPro = StandardScaler()
X_testPcaPro = pcaPro.fit_transform(X_testNum)

#PCA para reduzir as colunas em 2 componentes
pca_transformer = PCA(n_components=2)
X_testPca = pca_transformer.fit_transform(X_testPcaPro)

pcaTest11['PCA1'] = X_testPca[:, 0]
pcaTest11['PCA2'] = X_testPca[:, 1]

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=pcaTest11['PCA1'],               #componente principal 1 (c/ maior variação)
    y=pcaTest11['PCA2'],               #componente principal 2 (c/ 2 maior variação)
    hue=pcaTest11['CLUSTER'],
    palette='tab20',
    s=100,
    alpha=0.8
)

plt.title('Distribuição dos clusters encontrados com K-Médias', fontsize=16)
plt.xlabel('Componente principal 1', fontsize=12)
plt.ylabel('Componente principal 2', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Cluster')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

A partir desse gráfico podemos observar que há fortes indícios de que a maior parte das culturas compartilha características semelhantes. Ainda sim, podemos implicar que:


*   Clusters 2 e 7 são distintos da maioria, mas semelhantes entre si
*   Clusters 3 e 10 são bem distintos da maioria
*   Cluster 5 é bem esparso, o que sugere que ele é pouco coeso
*   Cluster 1 é denso, o que sugere que seus pontos têm alta semelhança



### Culturas por cluster

In [ ]:
cmap_22 = plt.cm.get_cmap('tab20', 23)
fig = plt.figure(figsize=(30,6))

plot_df_data_test = pd.DataFrame({'CLUSTER': testKM['CLUSTER'], 'label_encoded': y_test.reset_index(drop=True)})
plot_df_data_test['label'] = encoder.inverse_transform(plot_df_data_test['label_encoded'])

plot_df_test = pd.crosstab(index=plot_df_data_test['CLUSTER'], columns=plot_df_data_test['label'], normalize='index')

ax = fig.add_subplot(1, 3, 1)
plot_df_test.plot.bar(stacked=True, ax=ax, alpha=0.6, cmap=cmap_22)
ax.set_title('Culturas por cluster', alpha=0.5)

ax.set_ylim(0, 1.4)
ax.legend(bbox_to_anchor=(1.05, 1),
            loc='upper left',
            borderaxespad=0.,
            frameon=False,
            fontsize=11)
plt.subplots_adjust(right=0.85)
ax.xaxis.grid(False)


plt.tight_layout()
plt.show()

### Clusters analisados pelas colunas numéricas

In [ ]:
X_testOG = scaler.inverse_transform(X_test)

testKMOG = pd.DataFrame(X_testOG, columns=X.columns)
testKMOG['CLUSTER'] = testClusters + 1

numColTestKMOG = testKMOG.select_dtypes(include=np.number).drop(['CLUSTER'], axis=1).columns

cols = 2
fils = (len(numColTestKMOG) + cols - 1) // cols

fig = plt.figure(figsize=(10, 5 * fils))

for i, column in enumerate(numColTestKMOG):
  plotTestOG = testKMOG.groupby('CLUSTER')[column].mean()


  ax = fig.add_subplot(fils, cols, i + 1)

  ax.bar(plotTestOG.index, plotTestOG, color=sns.color_palette('Set2'), alpha=0.7)
  ax.set_title(f'{column.replace('_', ' ').title()} por cluster', alpha=0.8)
  ax.set_xlabel('Cluster')
  ax.set_ylabel(f'Média de {column.replace('_', ' ').title()}')
  ax.grid(False)
  ax.axhline(0.0, color='grey', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()

## 2.7 Treino do modelo com 3, 6 e 12 clusters

### Treino com 3

In [ ]:
k3 = KMeans(n_clusters=3, random_state=42)
k3.fit(X_train, y_train)

trainKM3 = pd.DataFrame(X_train, columns=X.columns)
testKM3 = pd.DataFrame(X_test, columns=X.columns)

trainCl3 = k3.predict(X_train)
testCl3 = k3.predict(X_test)


trainKM3['CLUSTER3'] = trainCl3 + 1
testKM3['CLUSTER3'] = testCl3 + 1


In [ ]:
pcaTest3 = testKM3

colNumTest3 = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X_testNum3 = pcaTest3[colNumTest3]

pcaPro3 = StandardScaler()
X_testPcaPro3 = pcaPro3.fit_transform(X_testNum3)

pca_transformer3 = PCA(n_components=2)
X_testPca3 = pca_transformer3.fit_transform(X_testPcaPro3)

pcaTest3['PCA1'] = X_testPca3[:, 0]
pcaTest3['PCA2'] = X_testPca3[:, 1]

plt.figure(figsize=(10, 5))
sns.scatterplot(
    x=pcaTest3['PCA1'],
    y=pcaTest3['PCA2'],
    hue=pcaTest3['CLUSTER3'],
    palette='tab20',
    s=100,
    alpha=0.8
)

plt.title('Distribuição dos clusters encontrados com K-Médias', fontsize=16)
plt.xlabel('Componente principal 1', fontsize=12)
plt.ylabel('Componente principal 2', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Cluster')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

###Treino com 6

In [ ]:
k6 = KMeans(n_clusters=6, random_state=42)
k6.fit(X_train, y_train)

trainKM6 = pd.DataFrame(X_train, columns=X.columns)
testKM6 = pd.DataFrame(X_test, columns=X.columns)

trainC6 = k6.predict(X_train)
testC6 = k6.predict(X_test)


trainKM6['CLUSTER6'] = trainC6 + 1
testKM6['CLUSTER6'] = testC6 + 1

In [ ]:
pcaTest6 = testKM6

colNumTest6 = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X_testNum6 = pcaTest6[colNumTest6]

pcaPro6 = StandardScaler()
X_testPcaPro6 = pcaPro6.fit_transform(X_testNum6)

pca_transformer6 = PCA(n_components=2)
X_testPca6 = pca_transformer6.fit_transform(X_testPcaPro6)

pcaTest6['PCA1'] = X_testPca6[:, 0]
pcaTest6['PCA2'] = X_testPca6[:, 1]

plt.figure(figsize=(10, 5))
sns.scatterplot(
    x=pcaTest6['PCA1'],
    y=pcaTest6['PCA2'],
    hue=pcaTest6['CLUSTER6'],
    palette='tab20',
    s=100,
    alpha=0.8
)

plt.title('Distribuição dos clusters encontrados com K-Médias', fontsize=16)
plt.xlabel('Componente principal 1', fontsize=12)
plt.ylabel('Componente principal 2', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Cluster')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

###Treino com 18

In [ ]:
k18 = KMeans(n_clusters=18, random_state=42)
k18.fit(X_train, y_train)


trainKM18 = pd.DataFrame(X_train, columns=X.columns)
testKM18 = pd.DataFrame(X_test, columns=X.columns)

trainC18 = k18.predict(X_train)
testC18 = k18.predict(X_test)

trainKM18['CLUSTER18'] = trainC18 + 1
testKM18['CLUSTER18'] = testC18 + 1

In [ ]:
pcaTest18 = testKM18

colNumTest18 = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X_testNum18 = pcaTest18[colNumTest18]

pcaPro18 = StandardScaler()
X_testPcaPro18 = pcaPro18.fit_transform(X_testNum18)

pca_transformer18 = PCA(n_components=2)
X_testPca18 = pca_transformer18.fit_transform(X_testPcaPro18)

pcaTest18['PCA1'] = X_testPca18[:, 0]
pcaTest18['PCA2'] = X_testPca18[:, 1]

plt.figure(figsize=(10, 5))
sns.scatterplot(
    x=pcaTest18['PCA1'],
    y=pcaTest18['PCA2'],
    hue=pcaTest18['CLUSTER18'],
    palette='tab20',
    s=100,
    alpha=0.8
)

plt.title('Distribuição dos clusters encontrados com K-Médias', fontsize=16)
plt.xlabel('Componente principal 1', fontsize=12)
plt.ylabel('Componente principal 2', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Cluster')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

###Comparação lado a lado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

#3 clusters
sns.scatterplot(
    x=pcaTest3['PCA1'],
    y=pcaTest3['PCA2'],
    hue=pcaTest3['CLUSTER3'],
    palette='tab20',
    s=100,
    alpha=0.8,
    ax=axes[0]
)
axes[0].set_title('Distribuição dos 3 clusters com K-Médias', fontsize=16)
axes[0].set_xlabel('Componente principal 1', fontsize=12)
axes[0].set_ylabel('Componente principal 2', fontsize=12)
axes[0].legend(title='Cluster')
axes[0].grid(True, linestyle='--', alpha=0.6)

#6 clusters
sns.scatterplot(
    x=pcaTest6['PCA1'],
    y=pcaTest6['PCA2'],
    hue=pcaTest6['CLUSTER6'],
    palette='tab20',
    s=100,
    alpha=0.8,
    ax=axes[1]
)
axes[1].set_title('Distribuição dos 6 clusters com K-Médias', fontsize=16)
axes[1].set_xlabel('Componente principal 1', fontsize=12)
axes[1].set_ylabel('Componente principal 2', fontsize=12)
axes[1].legend(title='Cluster')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

#11 clusters
sns.scatterplot(
    x=pcaTest11['PCA1'],
    y=pcaTest11['PCA2'],
    hue=pcaTest11['CLUSTER'],
    palette='tab20',
    s=100,
    alpha=0.8,
    ax=axes[0]
)
axes[0].set_title('Distribuição dos 11 clusters com K-Médias', fontsize=16)
axes[0].set_xlabel('Componente principal 1', fontsize=12)
axes[0].set_ylabel('Componente principal 2', fontsize=12)
axes[0].legend(title='Cluster')
axes[0].grid(True, linestyle='--', alpha=0.6)

#18 clusters
sns.scatterplot(
    x=pcaTest18['PCA1'],
    y=pcaTest18['PCA2'],
    hue=pcaTest18['CLUSTER18'],
    palette='tab20',
    s=100,
    alpha=0.8,
    ax=axes[1]
)
axes[1].set_title('Distribuição dos 18 clusters com K-Médias', fontsize=16)
axes[1].set_xlabel('Componente principal 1', fontsize=12)
axes[1].set_ylabel('Componente principal 2', fontsize=12)
axes[1].legend(title='Cluster')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

##2.8 max_iter

Parâmetro que limita a quantidade de iterações do algoritmo

In [ ]:
kMAX = KMeans(n_clusters=11, random_state=42, max_iter=2)
kMAX.fit(X_train, y_train)

trainMAX = pd.DataFrame(X_train, columns=X.columns)
testMAX = pd.DataFrame(X_test, columns=X.columns)

trClustersMAX = kMAX.predict(X_train)
testClustersMAX = kMAX.predict(X_test)


trainMAX['CLUSTER'] = trClustersMAX + 1
testMAX['CLUSTER'] = testClustersMAX + 1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(30, 8))

#sem max_iter
plot_df_test.plot.bar(stacked=True, ax=axes[0], alpha=0.6, cmap=cmap_22)
axes[0].set_title('Culturas por cluster sem max_iter', alpha=0.8)
axes[0].set_ylim(0, 1.4)
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., frameon=False, fontsize=11)
axes[0].xaxis.grid(False)

#max_iter = 2
plot_df_testMAX.plot.bar(stacked=True, ax=axes[1], alpha=0.6, cmap=cmap_22)
axes[1].set_title('Culturas por cluster com max_iter', alpha=0.8)
axes[1].set_ylim(0, 1.4)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., frameon=False, fontsize=11)
axes[1].xaxis.grid(False)

plt.tight_layout()
plt.show()

# 3. Neurônio Classificador de Multicamadas (Multi-Layer Perceptron Classifier)

1. Se trata de uma rede neural supervisionada (diferente do K-means)
2. Aprende a partir de dados corretos (como os de treino, com train_test_split)
3. Aprende com uma função não-linear que mapeia as entradas (features) para as saídas (targets/classes)
4. É um modelo de aprendizado de máquina (Machine Learning) e pode ser importado no Python por: **from sklearn.neural network import MLPClassifier**.

Antes de começar, é importante entender o que é um neurônio (perceptron), para então explicar o que é a rede neural Multi-Layer Perptron Classifire (MLPClassifier).

Neurônio (perceptron):



*   Um neurônio pode ser representado por uma entrada $x$, um peso $w$
e um bias (ativação) $b$, onde o valor de $z = w \cdot x + b$;
*   Também podem ser representados por um vetor de entradas $(x_1, . . . , x_n)$, um vetor de pesos $(w_1, . . . , w_n)$ e um vetor bias $b = (b_1, . . . , b_n)$;
*   Uma função de ativaçãoo (ReLU). Ela permite que a rede neural aprenda padrões complexos, algo que não é tão possível com apenas modelos lineares.

Uma rede neural se trata de vários neurônios trabalhando em conjunto.





In [ ]:
# O neuronio atribui o valor 1 a 'sim'
# O neuronio atribui o valor 0 a 'não'

# cada entrada pode ter um peso, atribuindo-se maior valor para eventos mais importante.

# por exemplo, se 'esta chovendo' posso atribuir peso 5, enquanto para 'foi previsto que ira chover' peso 2

def neuronio_guarda_chuva(entradas, peso,limiar):
  # Passo 1 e 2: multiplicar as entradas pelos respectivos pesos
  soma_ponderada = (entradas[0] * peso[0]) + (entradas[1] * peso[1])

  # Passo 3: tomar decisão
  if soma_ponderada > limiar:
    return 'levar o guarda-chuva'
  else:
    return 'não levar o guarda-chuva'

entradas = [0, 1]
pesos = [5 , 2]
limiar = 3

decisão = neuronio_guarda_chuva(entradas, pesos, limiar)
decisão

Multi-Layer (Multi Camadas):

*   **Camada de entrada**: Recebe os dados e o número de neurônios é igual
ao número de entradas (features);
*   **Camada(s) oculta(s)**: O cérebro da rede, onde os cálculos e a ativação
ocorrem (em várias camadas);
*   **Camada de saída**: Produz o resultado final. Geralmente tem um
neurônio por classe e usa uma função de ativação (softmax) que transforma saídas em probabilidades.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

X = data.drop(['label', 'label_encoded'], axis=1) # dados de entrada (feature)
y = data['label_encoded']                         # dados de saída (target)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

# 1. Instanciar e Treinar o MLPClassifier

# (100,) significa UMA camada oculta com 100 neurônios.
# (50, 30) significaria DUAS camadas, a primeira com 50 e a segunda com 30.
# (10,15,20) significa TRÊS camadas, a primeira com 10, a segunda com 15 e a terceira com 20.
mlp = MLPClassifier(hidden_layer_sizes=(50,30), # Duas camadas com 50 e 30 neurônios, respectivamente.
                    max_iter=500,               # Aumentar iterações para garantir convergência
                    activation='relu',          # Função de ativação padrão
                    solver='adam',              # O otimizador recomendado
                    random_state=42,
                    alpha=0.0001,               # Termo de regularização (para evitar overfitting)
                    learning_rate_init=0.001)   # Taxa de aprendizado inicial

# Treinar o modelo com os dados ESCALONADOS
mlp.fit(X_train, y_train)


# 2. Fazer Previsões e Avaliar
y_pred = mlp.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nAcurácia do Modelo: {accuracy * 100:.2f}%")

# Relatório detalhado
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

# (Código Bônus para visualizar a fronteira de decisão)
from mlxtend.plotting import plot_decision_regions

plt.figure(figsize=(8, 6))

# Calculate mean values for features to be used as filler values
# Assuming we want to plot the first two features (N and P after scaling)
# Features indices are 0 to 6. If plotting 0 and 1, filler_feature_values needed for 2, 3, 4, 5, 6.
mean_features = np.mean(X_test, axis=0)
filler_feature_values = {i: mean_features[i] for i in range(2, X_test.shape[1])}

# Use o X_test_scaled para a visualização
plot_decision_regions(X=X_test, y=y_test.values, clf=mlp, legend=2,
                      feature_index=[0, 1], filler_feature_values=filler_feature_values)
plt.title("Fronteira de Decisão do MLPClassifier")
plt.xlabel("Nitrogênio (N)")
plt.ylabel("Potássio (P)")
plt.show()

Aprendizado do MLPClassifier (“treinamento”): Um processo de duas fases que
repetem milhares de vezes




1.   **Fase 1**: Previsão (Fowardpropagation)
* É inserido um lote de dados na camada de entrada;
* Os dados “fluem” para frente, através das camadas ocultas;
* Cada neurônio faz seus próprios cálculos (soma ponderada +
ativação) e passa o resultado para a prõxima camada;
* A camada de saída produz uma previsão.

2. **Fase 2**: Correção (Backpropagation)
*  **Função de custo**: O algoritmo compara a previsão da rede
(model.predict(x teste)) com os corretos (y_teste), e usa
a uma função de custo para calcular o quão bem o modelo prevê os dados;
*  **Gradiente**: Cálcula o gradiente da função custo em relação ao
peso e o viés da rede neural (pense que o “erro” é o topo de
uma montanha; o gradiente aponta para a direção com maior
inclinação para baixo, a descida mais rápida);
*  **Backpropagation**: É o algoritmo que propaga o erro da camada de saída até a camada de entrada, descobrindo a contribuição de cada peso para o erro do modelo;
*  **Otimização**: É usado um otimizador para atualizar cada peso e viés, ajustando o modelo aos dados de treino (x_treino, y_treino), movendo-os na direção oposta ao do gradiente, pondo:
$$novo\_peso = peso\_antigo - (taxa\_de\_aprendizado * \nabla grad)$$

O algoritmo MLP repete ambas as Fases até que os pesos parem de mudar significativamente e o erro seja o menor possível.






FIM

In [ ]:
# (Código Bônus para visualizar a fronteira de decisão)
from mlxtend.plotting import plot_decision_regions
import matplotlib.pyplot as plt
import numpy as np

# AS VARIÁVEIS DE DADOS DEVEM ESTAR DEFINIDAS NO SEU AMBIENTE:
# X_test, y_test, mlp

# ===============================================================
# PASSO 1: DEFINIÇÃO DOS NOMES DAS FEATURES (AJUSTE AQUI!)
# ===============================================================
# Crie uma lista com os nomes das suas 7 features.
# Exemplo: Se seus dados são N, P, K, pH, etc.
FEATURE_NAMES = [
    "Nitrogênio (N)",
    "Fósforo (P)",
    "Potássio (K)",
    "pH do Solo",
    "Umidade",
    "Temperatura",
    "Luminosidade"
]
# Note que a lista deve ter o mesmo número de elementos que X_test.shape[1]

# ----------------- GERAÇÃO DOS PARES DE ÍNDICES -----------------
N_FEATURES = X_test.shape[1]
indexs_plot = []

for i in range(N_FEATURES):
    for j in range(i + 1, N_FEATURES):
        indexs_plot.append([i, j])

# Calcula as médias das features de uma vez
mean_features = np.mean(X_test, axis=0)

# ----------------- LOOP DE PLOTAGEM AUTOMATIZADO -----------------

print(f"Gerando {len(indexs_plot)} gráficos 2D...")

for feature_a_index, feature_b_index in indexs_plot:

    # BUSCA AUTOMÁTICA DOS NOMES
    name_a = FEATURE_NAMES[feature_a_index]
    name_b = FEATURE_NAMES[feature_b_index]

    # 2. Define o par de features para a plotagem atual
    FEATURES_TO_PLOT = [feature_a_index, feature_b_index]

    # 3. Identifica e calcula os valores médios para as features FIXAS (fillers)
    all_indices = range(N_FEATURES)
    filler_indices = [i for i in all_indices if i not in FEATURES_TO_PLOT]
    filler_feature_values = {i: mean_features[i] for i in filler_indices}

    # 4. Geração do Gráfico
    plt.figure(figsize=(8, 6))

    plot_decision_regions(X=X_test,
                          y=y_test.values,
                          clf=mlp,
                          legend=2,
                          feature_index=FEATURES_TO_PLOT,
                          filler_feature_values=filler_feature_values)

    # 5. RÓTULOS E TÍTULOS AUTOMATIZADOs

    # Título dinâmico
    plt.title(f"Fronteira de Decisão do MLP - {name_a} vs. {name_b}")

    # Rótulos automáticos usando o nome das features
    plt.xlabel(name_a)
    plt.ylabel(name_b)

    plt.show()

# 4 Análise dos Resultados
